# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:

1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: _my intuition was rough, the model found the real signal, and I can read exactly what it found._


## 0. Setup (Colab or local)

**Method, step by step:** the first cell finds the project's top folder (cloning it on Colab, where the notebook starts on an empty machine), loads the 30,000-page starter CSV, and defines the label: a page is _declining_ when its trend is down. The second cell loads the tools and defines `show()` and `show_table()`, so every result is a titled table in the same layout, plus the chart theme used by the one chart below.


In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

In [ ]:
import subprocess
import tempfile
import urllib.request

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

pd.set_option("display.width", 160)
NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))

# Emphasis theme: cividis navy for the story, grays for what should recede.
INK, MUTED, FAINT, GRID, SURFACE = "#111111", "#5f5e5a", "#9a9893", "#eeede9", "#ffffff"
NAVY, GRAY, PALE = mpl.colormaps["cividis"](0.05), "#9a9893", "#d4d2cc"


def load_inter() -> str:
    """Register the Inter font for this session and return the font family to draw with.

    The 400 and 600 weights are cached in the system temp folder, never in the project.
    If the download is unavailable, the chart falls back to matplotlib's default font,
    so the notebook still runs offline.
    """
    cache = os.path.join(tempfile.gettempdir(), "inter-font")
    os.makedirs(cache, exist_ok=True)
    try:
        for weight in (400, 600):
            path = os.path.join(cache, f"inter-{weight}.ttf")
            if not os.path.exists(path):
                url = f"https://cdn.jsdelivr.net/fontsource/fonts/inter@latest/latin-{weight}-normal.ttf"
                urllib.request.urlretrieve(url, path)
            font_manager.fontManager.addfont(path)
    except OSError:
        return "DejaVu Sans"
    return "Inter"


plt.rcParams.update({"font.family": load_inter()})

## 1. A rule you write by hand: `stale x visible`

Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

**Method:** this cell gives every page a score from the hand rule: it is zero unless the page is both stale and visible, and then it equals the page's impressions. It lists the ten highest-scoring pages and counts how many pages score above zero at all.


In [ ]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
show_table("The ten pages the hand rule scores highest", top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]])
show("How much of the data the hand rule can rank", {
    "pages with a score above zero": f"{(df['hand_rule_score'] > 0).sum():,}",
    "pages scored exactly zero (tied)": f"{(df['hand_rule_score'] == 0).sum():,}",
    "of the scored pages, how many are declining": f"{df.loc[df['hand_rule_score'] > 0, 'is_declining_label'].sum()} of {(df['hand_rule_score'] > 0).sum()}",
})

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.


**Method:** this cell defines `precision_at_k`. One detail matters: when many pages share the same score, the order among them is arbitrary, and a naive version would quietly let row order decide the answer. This version fills the slots left at the cutoff with the **average outcome of the tied pages**, so the number does not depend on how the table happens to be sorted. It then scores the hand rule at K = 20 and K = 50 and sets the base rate beside it.


In [ ]:
def precision_at_k(scores, labels, k: int) -> float:
    """Share of the top `k` pages that are declining, with tied scores resolved by their average.

    Pages with equal scores have no real order. The slots left at the cutoff are filled with the
    average label of the tied group, so the result does not depend on row order.
    """
    scores, labels = np.asarray(scores, dtype=float), np.asarray(labels, dtype=float)
    cutoff = np.sort(scores)[::-1][k - 1]
    above, tied = scores > cutoff, scores == cutoff
    return float((labels[above].sum() + (k - above.sum()) * labels[tied].mean()) / k)


def tied_slots(scores, k: int) -> int:
    """How many of the top `k` slots are filled by pages whose score is tied at the cutoff."""
    scores = np.asarray(scores, dtype=float)
    return int(k - (scores > np.sort(scores)[::-1][k - 1]).sum())


y = df["is_declining_label"].to_numpy()
rule_scores = df["hand_rule_score"].to_numpy()
show_table("The hand rule's precision on all 30,000 pages", pd.DataFrame({
    "Precision": {
        "Hand rule at K = 20": f"{precision_at_k(rule_scores, y, 20):.3f}",
        "Hand rule at K = 50": f"{precision_at_k(rule_scores, y, 50):.3f}",
        "Picking pages at random (base rate)": f"{y.mean():.3f}",
    },
    "Slots filled by tied pages": {
        "Hand rule at K = 20": f"{tied_slots(rule_scores, 20)} of 20",
        "Hand rule at K = 50": f"{tied_slots(rule_scores, 50)} of 50",
        "Picking pages at random (base rate)": "all",
    },
}).rename_axis("Scorer"))

## 2. Let a model learn the rule — then read it

A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.


In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.


## 3. Why you can't feed the outcome back in

Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:


In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable _before_ the outcome.

> Rule of thumb: if a feature would only be known _because someone already made the decision you're predicting_, it leaks. Leave it out.


## 4. 🔧 Your turn

- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored _in-sample_ here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.


In [ ]:
# Your experiment here


### Save your work

**Colab:** _File → Save a copy in GitHub_ (your submission) and _File → Save a copy in Drive_.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.
